In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import rc
from matplotlib.ticker import MaxNLocator

import optimal_consumption as oc

In [ ]:
rc("font", family="serif", serif=["Computer Modern"], size=18)
rc("text", usetex=True)

In [ ]:
R = 1.5

def delta(y):
    return 0.02


def r(y):
    return 0.0168


def lmbda(y):
    return y


def sigma(y):
    return 0.151


rho = -0.935

kappa = 0.271


def a(y):
    return -kappa * (y - 0.041 / 0.151)


nu = 0.0343 / 0.151


# nu = 1
def b(y):
    return np.full_like(y, nu, dtype=float)

In [ ]:
eta = oc.myopic_consumption_rate(R, r, delta, lmbda)

In [ ]:
X = np.linspace(-1, 1, 100)
for n in [1, 2]:
    X_sol, sol = oc.optimal_consumption_rate(
        R, r, delta, lmbda, rho, a, b, -n, n, 10000
    )
    plt.plot(
        X, 100 * np.interp(X, X_sol, sol, left=np.nan, right=np.nan), label=f"$m={n}$"
    )

X_sol, sol = oc.optimal_consumption_rate(R, r, delta, lmbda, rho, a, b, -3, 3, 10000)
plt.plot(
    X,
    100 * np.interp(X, X_sol, sol, left=np.nan, right=np.nan),
    linestyle="dotted",
    label="$m=3$",
)


# plt.plot(X, 100 * eta(X), color="grey", linestyle="dashed", label="$\\eta(y)$")

plt.legend()

plt.xlabel("$y$")
plt.ylabel("Consumption Rate (\\%)")

plt.savefig("img/mpr_ou/convergence_neumann.pdf", bbox_inches="tight")

In [ ]:
X = np.linspace(-1, 1, 100)
ns = range(1, 5)
u = None
diffs = []
for n in ns:
    X_sol, sol = oc.optimal_consumption_rate(
        R, r, delta, lmbda, rho, a, b, -n, n, 10000
    )

    eval = np.interp(X, X_sol, sol)

    if u is not None:
        diffs.append(np.linalg.norm(eval - u, np.inf))

    u = eval

plt.plot(ns[1:], np.log(diffs))

plt.xlabel("$m$")
plt.ylabel(r"$\log \| u_m - u_{m-1}\|_{L^\infty([-1, 1])}$")

plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))

plt.savefig("img/mpr_ou/convergence-rate_neumann.pdf", bbox_inches="tight")

In [ ]:
X = np.linspace(-5, 5, 100)
X_sol, sol = oc.optimal_consumption_rate(R, r, delta, lmbda, rho, a, b, -50, 50, 10000)
sol_pf = oc.optimal_investment_fraction(R, sigma, lmbda, rho, b, X_sol, sol)

plt.plot(X, 100 * np.interp(X, X_sol, sol_pf), label="Optimal")
plt.plot(X, 100 * oc.myopic_investment_fraction(R, sigma, lmbda)(X), label="Myopic")
plt.fill_between(
    X,
    100 * oc.myopic_investment_fraction(R, sigma, lmbda)(X),
    100 * np.interp(X, X_sol, sol_pf),
    color="green",
    alpha=0.3,
    hatch="//",
    label="Intertemporal Hedge",
)

plt.legend()

plt.xlabel("$y$")
plt.ylabel("Risky Asset Allocation (\\%)")

plt.savefig("img/mpr_ou/portfolio_limit.pdf", bbox_inches="tight")

In [ ]:
X = np.linspace(-25, 25, 1000)
X_sol, sol = oc.optimal_consumption_rate(R, r, delta, lmbda, rho, a, b, -50, 50, 10000)

plt.plot(X, 100 * (np.interp(X, X_sol, sol) - eta(X)), label="$\\xi_h$")
plt.axhline(
    100 * (-2 * (kappa + (R - 1) / R * rho * nu)),
    color="grey",
    linestyle="dashed",
    label="$-2 \\tilde{\\kappa}$",
)

plt.legend()

plt.xlabel("$y$")
plt.ylabel("Consumption Adjustment (\%)")

plt.savefig("img/mpr_ou/consumption_limit.pdf", bbox_inches="tight")

In [ ]:
X = np.linspace(-0.3, 0.9, 1000)
X_sol, sol = oc.optimal_consumption_rate(R, r, delta, lmbda, rho, a, b, -5, 5, 10000)
sol_pf = oc.optimal_investment_fraction(R, sigma, lmbda, rho, b, X_sol, sol)

plt.plot(X, 100 * np.interp(X, X_sol, sol), label="Optimal")
plt.plot(X, 100 * eta(X), label="Myopic")
plt.legend()
plt.xlabel("$y$")
plt.ylabel("Consumption Rate (\\%)")
plt.savefig("img/mpr_ou/consumption.pdf", bbox_inches="tight")
plt.show()

plt.axhline(0, linestyle="dashed", color="grey", alpha=0.7)
plt.axvline(0, linestyle="dashed", color="grey", alpha=0.7)

plt.plot(X, 100 * np.interp(X, X_sol, sol_pf), label="Optimal")
plt.plot(X, 100 * oc.myopic_investment_fraction(R, sigma, lmbda)(X), label="Myopic")
plt.fill_between(
    X,
    100 * oc.myopic_investment_fraction(R, sigma, lmbda)(X),
    100 * np.interp(X, X_sol, sol_pf),
    color="green",
    alpha=0.3,
    hatch="//",
    label="Intertemporal Hedge",
)

plt.ylabel("Risky Asset Allocation (\\%)")
plt.legend()
plt.xlabel("$y$")
plt.savefig("img/mpr_ou/portfolio.pdf", bbox_inches="tight")
plt.show()

In [ ]:
np.interp(0, X_sol, sol_pf)